In [5]:
class Vector:
    def __init__(self, data):
        self.data = list(data)
        self.size = len(self.data)

    def __repr__(self):
        return f"Vector({self.data})"

    def __add__(self, other):
        return Vector([a + b for a, b in zip(self.data, other.data)])

    def __sub__(self, other):
        return Vector([a - b for a, b in zip(self.data, other.data)])

    def __mul__(self, scalar):
        return Vector([x * scalar for x in self.data])
    
    def dot(self, other):
        return sum(a * b for a, b in zip(self.data, other.data))

    def magnitude(self):
        return sum(x ** 2 for x in self.data) ** 0.5

In [3]:
print(Vector([3, 4]) + Vector([1, 0]))
print(Vector([3, 4]) - Vector([1, 0]))
print(Vector([3, 4]) * 2)

Vector([4, 4])
Vector([2, 4])
Vector([6, 8])


In [6]:
print(Vector([3, 4]).dot(Vector([1, 0])))
print(Vector([3, 4]).magnitude())

3
5.0


In [47]:
class Matrix:
    def __init__(self, data):
        self.data = [list(row) for row in data]
        self.rows = len(self.data)
        self.cols = len(self.data[0])
        self.shape = (self.rows, self.cols)

    def __repr__(self):
        rows_str = "\n  ".join(str(row) for row in self.data)
        return f"Matrix({self.shape}):\n  {rows_str}"

    def __add__(self, other):
        return Matrix([
            [self.data[i][j] + other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)
        ])
    
    def __sub__(self, other):
        return Matrix([
            [self.data[i][j] - other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)
        ])

    def scalar_multiply(self, scalar):
        return Matrix([
            [self.data[i][j] * scalar for j in range(self.cols)]
            for i in range(self.rows)
        ])

    def element_wise_multiply(self, other):
        return Matrix([
            [self.data[i][j] * other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)
        ])

    def matmul(self, other):
        return Matrix([
            [
                sum(self.data[i][k] * other.data[k][j] for k in range(self.cols))
                for j in range(other.cols)
            ]
            for i in range(self.rows)
        ])

    def transpose(self):
        return Matrix([
            [self.data[j][i] for j in range(self.rows)]
            for i in range(self.cols)
        ])

    def determinant(self):
        if self.shape == (1, 1):
            return self.data[0][0]
        if self.shape == (2, 2):
            return self.data[0][0] * self.data[1][1] - self.data[0][1] * self.data[1][0]
        det = 0
        for j in range(self.cols):
            minor = Matrix([
                [self.data[i][k] for k in range(self.cols) if k != j]
                for i in range(1, self.rows)
            ])
            det += ((-1) ** j) * self.data[0][j] * minor.determinant()
        return det

    def inverse_2x2(self):
        det = self.determinant()
        if det == 0:
            raise ValueError("Matrix is singular, no inverse exists")
        return Matrix([
            [self.data[1][1] / det, -self.data[0][1] / det],
            [-self.data[1][0] / det, self.data[0][0] / det]
        ])

    @staticmethod
    def identity(n):
        return Matrix([
            [1 if i == j else 0 for j in range(n)]
            for i in range(n)
        ])

    def minor(self, drop_row, drop_col):
        return Matrix([
            [self.data[i][j] for j in range(self.cols) if j != drop_col]
            for i in range(self.rows) if i != drop_row
        ])
    
    def inverse_3x3(self):
        if self.shape != (3, 3):
            raise ValueError("inverse_3x3 requires a 3x3 matrix")
        det = self.determinant()
        if det == 0:
            raise ValueError("Matrix is singular, no inverse exists")
        cofactors = []
        for i in range(3):
            row = []
            for j in range(3):
                sign = (-1) ** (i + j)
                row.append(sign * self.minor(i, j).determinant())
            cofactors.append(row)
        adjugate = Matrix(cofactors).transpose()
        return adjugate.scalar_multiply(1 / det)

    

In [28]:
A = Matrix([[1, 2], [3, 4]])
B = Matrix([[5, 6], [7, 8]])
print(A.shape, B.shape)
print((A + B).data)

(2, 2) (2, 2)
[[6, 8], [10, 12]]


In [29]:
A = Matrix([[1, 2], [3, 4]])
B = Matrix([[5, 6], [7, 8]])
print((A - B).data)
print(A.scalar_multiply(10).data)

[[-4, -4], [-4, -4]]
[[10, 20], [30, 40]]


In [30]:
C = Matrix([[1, 0], [2, 3]])
D = Matrix([[4, 5], [6, 7]])
print(C.element_wise_multiply(D).data)

[[4, 0], [12, 21]]


In [31]:
C = Matrix([[1, 0], [2, 3]])
D = Matrix([[4, 5], [6, 7]])
print(C.element_wise_multiply(D).data)
print(C.matmul(D).data)

[[4, 0], [12, 21]]
[[4, 5], [26, 31]]


In [32]:
M = Matrix([[1, 2, 3], [4, 5, 6]])
print(M.shape)
print(M.transpose().shape)
print(M.transpose().data)

(2, 3)
(3, 2)
[[1, 4], [2, 5], [3, 6]]


In [33]:
print(Matrix([[1, 2], [3, 4]]).determinant())
print(Matrix([[1, 2], [2, 4]]).determinant())

-2
0


In [34]:
A = Matrix([[1, 2], [3, 4]])
print(A.inverse_2x2().data)

[[-2.0, 1.0], [1.5, -0.5]]


In [37]:
A = Matrix([[1, 2], [3, 4]])
print(Matrix.identity(2).data)
print(A.matmul(A.inverse_2x2()).data)

[[1, 0], [0, 1]]
[[1.0, 0.0], [0.0, 1.0]]


In [39]:
A = Matrix([[1, 2], [3, 4]])
B = Matrix([[5, 6], [7, 8]])

print("A + B =", (A + B).data)
print("A @ B =", A.matmul(B).data)
print("A^T =", A.transpose().data)
print("det(A) =", A.determinant())
print("A^-1 =", A.inverse_2x2().data)

I = Matrix.identity(2)
print("A @ A^-1 =", A.matmul(A.inverse_2x2()).data)

A + B = [[6, 8], [10, 12]]
A @ B = [[19, 22], [43, 50]]
A^T = [[1, 3], [2, 4]]
det(A) = -2
A^-1 = [[-2.0, 1.0], [1.5, -0.5]]
A @ A^-1 = [[1.0, 0.0], [0.0, 1.0]]


In [40]:
import random

inputs = Matrix([[0.5], [0.8], [0.2]])
weights = Matrix([
    [random.uniform(-1, 1) for _ in range(3)] for _ in range(2)
])
bias = Matrix([[0.1], [0.1]])

def relu_matrix(m):
    return Matrix([[max(0, val) for val in row] for row in m.data])

pre_activation = weights.matmul(inputs) + bias
output = relu_matrix(pre_activation)

print(f"Input shape: {inputs.shape}")
print(f"Weight shape: {weights.shape}")
print(f"Output shape: {output.shape}")
print(f"Output: {output.data}")

Input shape: (3, 1)
Weight shape: (2, 3)
Output shape: (2, 1)
Output: [[0], [0]]


In [45]:
import numpy as np

A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print("A + B =\n", A + B)
print("A * B (element-wise) =\n", A * B)
print("A @ B (matrix multiply) =\n", A @ B)
print("A^T =\n", A.T)
print("det(A) =", np.linalg.det(A))
print("A^-1 =\n", np.linalg.inv(A))
print("I =\n", np.eye(2))

inputs = np.random.randn(3, 1)
weights = np.random.randn(2, 3)
bias = np.array([[0.1], [0.1]])
output = np.maximum(0, weights @ inputs + bias)

print(f"\nNeural network layer: {weights.shape} @ {inputs.shape} = {output.shape}")
print(f"Output:\n{output}")

matrix = np.array([[1, 2, 3], [4, 5, 6]])
bias_row = np.array([10, 20, 30])
print(matrix + bias_row)

A + B =
 [[ 6  8]
 [10 12]]
A * B (element-wise) =
 [[ 5 12]
 [21 32]]
A @ B (matrix multiply) =
 [[19 22]
 [43 50]]
A^T =
 [[1 3]
 [2 4]]
det(A) = -2.0000000000000004
A^-1 =
 [[-2.   1. ]
 [ 1.5 -0.5]]
I =
 [[1. 0.]
 [0. 1.]]

Neural network layer: (2, 3) @ (3, 1) = (2, 1)
Output:
[[0.]
 [0.]]
[[11 22 33]
 [14 25 36]]


# test

In [46]:
# 练习 1：验证逆矩阵
candidates = [
    Matrix([[1, 2], [3, 4]]),
    Matrix([[2, 0], [0, 3]]),
    Matrix([[4, 1], [2, 3]]),
    Matrix([[1, 2], [2, 4]]),
]
for M in candidates:
    print("det =", M.determinant())
    try:
        print(M.matmul(M.inverse_2x2()).data)
    except ValueError as err:
        print(type(err).__name__, str(err))

det = -2
[[1.0, 0.0], [0.0, 1.0]]
det = 6
[[1.0, 0.0], [0.0, 1.0]]
det = 10
[[1.0, 0.0], [-1.1102230246251565e-16, 1.0000000000000002]]
det = 0
ValueError Matrix is singular, no inverse exists


In [ ]:
# 实现 3x3 逆矩阵
def minor(self, drop_row, drop_col):
    return Matrix([
        [self.data[i][j] for j in range(self.cols) if j != drop_col]
        for i in range(self.rows) if i != drop_row
    ])

def inverse_3x3(self):
    if self.shape != (3, 3):
        raise ValueError("inverse_3x3 requires a 3x3 matrix")
    det = self.determinant()
    if det == 0:
        raise ValueError("Matrix is singular, no inverse exists")
    cofactors = []
    for i in range(3):
        row = []
        for j in range(3):
            sign = (-1) ** (i + j)
            row.append(sign * self.minor(i, j).determinant())
        cofactors.append(row)
    adjugate = Matrix(cofactors).transpose()
    return adjugate.scalar_multiply(1 / det)

In [48]:
M = Matrix([[1, 2, 3], [0, 1, 4], [5, 6, 0]])
print("det", M.determinant())
mine = np.array(M.inverse_3x3().data, dtype=float)
ref = np.linalg.inv(np.array(M.data, dtype=float))
print(np.allclose(mine, ref))

det 1
True


In [50]:
# 搭建双层神经网络

random.seed(42)

x = Matrix([[0.5], [0.8], [0.2]])
W1 = Matrix([random.uniform(-1, 1) for _ in range(3)] for _ in range(4))
b1 = Matrix([[0.0] for _ in range(4)])
W2 = Matrix([random.uniform(-1, 1) for _ in range(4)] for _ in range(2))
b2 = Matrix([[0.0] for _ in range(2)])

z1 = W1.matmul(x) + b1
h1 = relu_matrix(z1)
z2 = W2.matmul(h1) + b2

print(x.shape, W1.shape, b1.shape)
print(z1.shape, h1.shape)
print(W2.shape, b2.shape, z2.shape)

(3, 1) (4, 3) (4, 1)
(4, 1) (4, 1)
(2, 4) (2, 1) (2, 1)
